In [ ]:
# ============================================================
# REALISED MODEL COMPARISON
# RF + XGBoost
# lambda = 0.5
#
# Purpose:
#
#   RF / XGBoost predictions
#          ↓
#   MILP optimisation
#          ↓
#   Planned schedule
#          ↓
#   ACTUAL_DURATION
#          ↓
#   Delay propagation
#          ↓
#   Realised operational performance
#
# IMPORTANT:
#
# Both models use:
#   - identical 12-case cohort
#   - identical observed actual durations
#   - identical MILP formulation
#   - identical lambda
#   - identical room capacity
#   - identical turnover
#
# ============================================================


import pandas as pd
import numpy as np
import os
import time


# ============================================================
# 1. LOAD MILP ENGINE
# ============================================================

%run milp_engine.ipynb


# ============================================================
# 2. GLOBAL SETTINGS
# ============================================================

MODELS = {

    "RF":
        "Prescriptive/RF_prescriptive_optimizer_inputs.csv",

    "XGBoost":
        "Prescriptive/XGBoost_prescriptive_optimizer_inputs.csv"

}


LAMBDA = 0.5

N_ROOMS = 3

ROOM_CAPACITY = 480.0

TURNOVER = 20.0

TIME_LIMIT = 60


COHORT_PATH = (
    "Cohort/fixed_cohort.csv"
)


ACTUAL_DATA_PATH = (
    "new_data/"
    "mover_epic_final_test_features.csv"
)


OUTPUT_DIR = "Results"


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 3. LOAD FIXED COHORT
# ============================================================

cohort_df = pd.read_csv(
    COHORT_PATH
)


cohort_ids = (
    cohort_df["LOG_ID"]
    .tolist()
)


N_CASES = len(
    cohort_ids
)


print("=" * 70)
print("FIXED COHORT")
print("=" * 70)

print(
    "Number of cases:",
    N_CASES
)

print(
    "Lambda:",
    LAMBDA
)

print(
    "Number of rooms:",
    N_ROOMS
)

print(
    "Room capacity:",
    ROOM_CAPACITY
)

print(
    "Turnover:",
    TURNOVER
)


# ============================================================
# 4. LOAD OBSERVED ACTUAL DURATIONS
# ============================================================

test_df = pd.read_csv(
    ACTUAL_DATA_PATH
)


if (
    "ACTUAL_DURATION"
    not in test_df.columns
):

    raise ValueError(
        "ACTUAL_DURATION column "
        "not found."
    )


actual_df = (

    test_df[
        test_df["LOG_ID"].isin(
            cohort_ids
        )
    ]

    [
        [
            "LOG_ID",
            "ACTUAL_DURATION"
        ]
    ]

    .copy()

)


# Preserve fixed cohort order

cohort_order = {

    log_id: i

    for i, log_id
    in enumerate(cohort_ids)

}


actual_df["Surgery"] = (

    actual_df["LOG_ID"]
    .map(cohort_order)

)


actual_df = (

    actual_df

    .sort_values(
        "Surgery"
    )

    .reset_index(
        drop=True
    )

)


# ============================================================
# 5. ACTUAL DURATION VALIDATION
# ============================================================

if len(actual_df) != N_CASES:

    raise ValueError(

        "Actual-duration cohort "
        "does not match fixed cohort."

    )


if actual_df[
    "ACTUAL_DURATION"
].isna().any():

    raise ValueError(

        "Missing actual durations "
        "found."

    )


if (
    actual_df[
        "ACTUAL_DURATION"
    ] <= 0
).any():

    raise ValueError(

        "Non-positive actual "
        "duration found."

    )


actual_map = dict(

    zip(

        actual_df["Surgery"],

        actual_df[
            "ACTUAL_DURATION"
        ]

    )

)


print("\n")
print("=" * 70)
print("ACTUAL DURATIONS")
print("=" * 70)

print(

    actual_df[
        [
            "Surgery",
            "LOG_ID",
            "ACTUAL_DURATION"
        ]
    ].to_string(
        index=False
    )

)


# ============================================================
# 6. HELPER FUNCTION:
#    PREPARE MODEL COHORT
# ============================================================

def prepare_model_cohort(
    model_path
):

    df = pd.read_csv(
        model_path
    )


    # Check required columns

    required_columns = [

        "LOG_ID",

        "DURATION_P50_MINS",

        "DURATION_P90_MINS"

    ]


    missing = [

        c

        for c in required_columns

        if c not in df.columns

    ]


    if missing:

        raise ValueError(

            f"Missing columns: "
            f"{missing}"

        )


    df_day = (

        df[
            df["LOG_ID"].isin(
                cohort_ids
            )
        ]

        .copy()

    )


    # Preserve identical cohort order

    df_day["LOG_ID"] = pd.Categorical(

        df_day["LOG_ID"],

        categories=cohort_ids,

        ordered=True

    )


    df_day = (

        df_day

        .sort_values(
            "LOG_ID"
        )

        .reset_index(
            drop=True
        )

    )


    if len(df_day) != N_CASES:

        raise ValueError(

            "Model cohort mismatch."

        )


    return df_day


# ============================================================
# 7. HELPER FUNCTION:
#    EXTRACT MILP SCHEDULE
# ============================================================

def extract_schedule_from_result(
    result
):

    """
    Extract room assignment and start
    times from the result returned by
    the updated solve_or_milp().

    Expected keys:

        Room_Assignment
        Start_Times

    or:

        room_assignment
        start_times
    """


    # --------------------------------------------------------
    # Try lowercase keys
    # --------------------------------------------------------

    if (
        "room_assignment"
        in result
    ):

        room_assignment = (
            result[
                "room_assignment"
            ]
        )

    elif (
        "Room_Assignment"
        in result
    ):

        room_assignment = (
            result[
                "Room_Assignment"
            ]
        )

    else:

        raise KeyError(

            "solve_or_milp() does not "
            "return room assignment. "
            "Modify milp_engine.ipynb "
            "to return room_assignment."

        )


    # --------------------------------------------------------
    # Start times
    # --------------------------------------------------------

    if (
        "start_times"
        in result
    ):

        start_times = (
            result[
                "start_times"
            ]
        )

    elif (
        "Start_Times"
        in result
    ):

        start_times = (
            result[
                "Start_Times"
            ]
        )

    else:

        raise KeyError(

            "solve_or_milp() does not "
            "return start times. "
            "Modify milp_engine.ipynb "
            "to return start_times."

        )


    return (
        room_assignment,
        start_times
    )


# ============================================================
# 8. HELPER FUNCTION:
#    REALISED SCHEDULE EVALUATION
# ============================================================

def evaluate_realised_schedule(

    model_name,

    room_assignment,

    start_times

):


    # --------------------------------------------------------
    # Build room cases
    # --------------------------------------------------------

    room_cases = {

        room: []

        for room in range(
            N_ROOMS
        )

    }


    for surgery in range(
        N_CASES
    ):

        room = int(
            room_assignment[
                surgery
            ]
        )


        if room not in room_cases:

            raise ValueError(

                f"Invalid room "
                f"{room} for surgery "
                f"{surgery}."

            )


        room_cases[
            room
        ].append(
            surgery
        )


    # --------------------------------------------------------
    # Sort by planned start
    # --------------------------------------------------------

    for room in room_cases:

        room_cases[
            room
        ].sort(

            key=lambda i:
            float(
                start_times[i]
            )

        )


    # --------------------------------------------------------
    # Delay propagation
    # --------------------------------------------------------

    realised_records = []


    for room in range(
        N_ROOMS
    ):


        previous_finish = None


        for surgery in room_cases[
            room
        ]:


            planned_start = float(

                start_times[
                    surgery
                ]

            )


            actual_duration = float(

                actual_map[
                    surgery
                ]

            )


            # First surgery in room

            if previous_finish is None:

                realised_start = (
                    planned_start
                )


            # Subsequent surgeries

            else:

                realised_start = max(

                    planned_start,

                    previous_finish
                    +
                    TURNOVER

                )


            realised_finish = (

                realised_start
                +
                actual_duration

            )


            start_delay = (

                realised_start
                -
                planned_start

            )


            realised_records.append({

                "Model":
                    model_name,

                "Surgery":
                    surgery,

                "LOG_ID":
                    cohort_ids[
                        surgery
                    ],

                "Room":
                    room,

                "Planned_Start":
                    planned_start,

                "Realised_Start":
                    realised_start,

                "Actual_Duration":
                    actual_duration,

                "Realised_Finish":
                    realised_finish,

                "Start_Delay":
                    start_delay

            })


            previous_finish = (
                realised_finish
            )


    realised_df = pd.DataFrame(
        realised_records
    )


    realised_df = (

        realised_df

        .sort_values(
            [
                "Room",
                "Realised_Start"
            ]
        )

        .reset_index(
            drop=True
        )

    )


    # --------------------------------------------------------
    # Room-level metrics
    # --------------------------------------------------------

    room_results = []


    for room in range(
        N_ROOMS
    ):


        room_df = (

            realised_df[
                realised_df["Room"]
                == room
            ]

            .sort_values(
                "Realised_Start"
            )

        )


        finish = (

            room_df[
                "Realised_Finish"
            ].max()

            if len(room_df) > 0

            else 0.0

        )


        busy = (

            room_df[
                "Actual_Duration"
            ].sum()

        )


        overtime = max(

            finish
            -
            ROOM_CAPACITY,

            0.0

        )


        total_delay = (

            room_df[
                "Start_Delay"
            ].sum()

        )


        max_delay = (

            room_df[
                "Start_Delay"
            ].max()

            if len(room_df) > 0

            else 0.0

        )


        room_results.append({

            "Model":
                model_name,

            "Room":
                room,

            "Finish":
                finish,

            "Actual_Busy_Time":
                busy,

            "Overtime":
                overtime,

            "Total_Start_Delay":
                total_delay,

            "Maximum_Start_Delay":
                max_delay

        })


    room_results_df = pd.DataFrame(
        room_results
    )


    # --------------------------------------------------------
    # System metrics
    # --------------------------------------------------------

    realised_makespan = (

        realised_df[
            "Realised_Finish"
        ].max()

    )


    realised_overtime = (

        room_results_df[
            "Overtime"
        ].sum()

    )


    total_actual_duration = (

        realised_df[
            "Actual_Duration"
        ].sum()

    )


    nominal_capacity = (

        N_ROOMS
        *
        ROOM_CAPACITY

    )


    nominal_capacity_load_ratio = (

        total_actual_duration
        /
        nominal_capacity

    )


    realised_horizon_utilisation = (

        total_actual_duration
        /
        (
            N_ROOMS
            *
            realised_makespan
        )

    )


    realised_evaluation_objective = (

        realised_overtime

        +

        0.5
        *
        realised_makespan

    )


    summary = {

        "Model":
            model_name,

        "Lambda":
            LAMBDA,

        "Number_of_Cases":
            N_CASES,

        "Realised_Makespan":
            realised_makespan,

        "Realised_Overtime":
            realised_overtime,

        "Total_Actual_Duration":
            total_actual_duration,

        "Nominal_Capacity_Load_Ratio":
            nominal_capacity_load_ratio,

        "Realised_Horizon_Utilisation":
            realised_horizon_utilisation,

        "Realised_Evaluation_Objective":
            realised_evaluation_objective

    }


    return (

        realised_df,

        room_results_df,

        summary

    )


# ============================================================
# 9. MAIN EXPERIMENT
# ============================================================

all_summaries = []


for model_name, model_path in MODELS.items():


    print("\n")
    print("=" * 70)
    print(
        f"RUNNING {model_name} "
        f"+ lambda = {LAMBDA}"
    )
    print("=" * 70)


    # --------------------------------------------------------
    # Load model cohort
    # --------------------------------------------------------

    df_day = prepare_model_cohort(
        model_path
    )


    print(
        f"Model: {model_name}"
    )

    print(
        f"Number of cases: "
        f"{len(df_day)}"
    )


    print("\nPrediction inputs:")

    print(

        df_day[
            [
                "LOG_ID",
                "DURATION_P50_MINS",
                "DURATION_P90_MINS"
            ]
        ].to_string(
            index=False
        )

    )


    # --------------------------------------------------------
    # Run MILP
    # --------------------------------------------------------

    solve_start = time.time()


    result = solve_or_milp(

        df_day=df_day,

        lam=LAMBDA

    )


    elapsed = (
        time.time()
        -
        solve_start
    )


    if result is None:

        raise RuntimeError(

            f"MILP failed for "
            f"{model_name}."

        )


    print("\n")
    print("MILP RESULT")
    print("-" * 70)


    print(
        "Objective:",
        result["Objective"]
    )

    print(
        "Overtime:",
        result["Overtime"]
    )

    print(
        "Makespan:",
        result["Makespan"]
    )

    print(
        "Solve Time:",
        elapsed
    )


    if "Status" in result:

        print(
            "Status:",
            result["Status"]
        )


    if "Gap" in result:

        print(
            "MIP Gap:",
            result["Gap"]
        )


    # --------------------------------------------------------
    # Extract schedule
    # --------------------------------------------------------

    (
        room_assignment,
        start_times
    ) = extract_schedule_from_result(
        result
    )


    # --------------------------------------------------------
    # Evaluate against ACTUAL_DURATION
    # --------------------------------------------------------

    (
        realised_df,
        room_results_df,
        summary

    ) = evaluate_realised_schedule(

        model_name,

        room_assignment,

        start_times

    )


    # --------------------------------------------------------
    # Print realised schedule
    # --------------------------------------------------------

    print("\n")
    print("=" * 70)
    print(
        f"{model_name.upper()} "
        "REALISED SCHEDULE"
    )
    print("=" * 70)


    print(

        realised_df[
            [
                "Surgery",
                "LOG_ID",
                "Room",
                "Planned_Start",
                "Realised_Start",
                "Actual_Duration",
                "Realised_Finish",
                "Start_Delay"
            ]
        ].to_string(
            index=False
        )

    )


    # --------------------------------------------------------
    # Room results
    # --------------------------------------------------------

    print("\n")
    print("=" * 70)
    print(
        f"{model_name.upper()} "
        "ROOM RESULTS"
    )
    print("=" * 70)


    print(

        room_results_df.to_string(
            index=False
        )

    )


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print("\n")
    print("=" * 70)
    print(
        f"{model_name.upper()} "
        "REALISED PERFORMANCE"
    )
    print("=" * 70)


    print(

        f"Realised Makespan: "
        f"{summary['Realised_Makespan']:.2f}"

    )


    print(

        f"Realised Overtime: "
        f"{summary['Realised_Overtime']:.2f}"

    )


    print(

        f"Total Actual Surgery Time: "
        f"{summary['Total_Actual_Duration']:.2f}"

    )


    print(

        f"Nominal Capacity Load Ratio: "
        f"{summary['Nominal_Capacity_Load_Ratio']:.4f}"

    )


    print(

        f"Realised-Horizon Utilisation: "
        f"{summary['Realised_Horizon_Utilisation']:.4f}"

    )


    print(

        f"Realised Evaluation Objective: "
        f"{summary['Realised_Evaluation_Objective']:.2f}"

    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    validation_errors = []


    for room in range(
        N_ROOMS
    ):


        room_df = (

            realised_df[
                realised_df["Room"]
                == room
            ]

            .sort_values(
                "Realised_Start"
            )

        )


        previous_finish = None


        for _, row in room_df.iterrows():


            if previous_finish is not None:

                required_start = (

                    previous_finish
                    +
                    TURNOVER

                )


                if (

                    row[
                        "Realised_Start"
                    ]

                    <

                    required_start
                ):

                    validation_errors.append(

                        f"Room {room}: "
                        f"Surgery "
                        f"{int(row['Surgery'])} "
                        f"overlap."

                    )


            previous_finish = (

                row[
                    "Realised_Finish"
                ]

            )


    if len(validation_errors) == 0:

        print("\nValidation: PASS")

    else:

        print("\nValidation: FAIL")

        for error in validation_errors:

            print(
                "ERROR:",
                error
            )


    # --------------------------------------------------------
    # Save model-specific files
    # --------------------------------------------------------

    model_tag = (

        "RF"

        if model_name == "RF"

        else

        "XGBoost"

    )


    schedule_path = (

        f"{OUTPUT_DIR}/"
        f"realised_{model_tag}"
        f"_lambda05_schedule.csv"

    )


    rooms_path = (

        f"{OUTPUT_DIR}/"
        f"realised_{model_tag}"
        f"_lambda05_room_results.csv"

    )


    summary_path = (

        f"{OUTPUT_DIR}/"
        f"realised_{model_tag}"
        f"_lambda05_summary.csv"

    )


    realised_df.to_csv(

        schedule_path,

        index=False

    )


    room_results_df.to_csv(

        rooms_path,

        index=False

    )


    pd.DataFrame(
        [summary]
    ).to_csv(

        summary_path,

        index=False

    )


    all_summaries.append(
        summary
    )


    print("\nSaved:")

    print(
        schedule_path
    )

    print(
        rooms_path
    )

    print(
        summary_path
    )


# ============================================================
# 10. FINAL COMPARISON
# ============================================================

comparison_df = pd.DataFrame(
    all_summaries
)


comparison_path = (

    f"{OUTPUT_DIR}/"
    "realised_RF_XGBoost_lambda05_comparison.csv"

)


comparison_df.to_csv(

    comparison_path,

    index=False

)


print("\n")
print("=" * 70)
print("FINAL RF vs XGBOOST REALISED COMPARISON")
print("=" * 70)


print(

    comparison_df[
        [
            "Model",
            "Lambda",
            "Realised_Makespan",
            "Realised_Overtime",
            "Total_Actual_Duration",
            "Nominal_Capacity_Load_Ratio",
            "Realised_Horizon_Utilisation",
            "Realised_Evaluation_Objective"
        ]
    ].to_string(
        index=False
    )

)


print("\nSaved:")

print(
    comparison_path
)

print("\n")
print("=" * 70)
print("EXPERIMENT FINISHED")
print("=" * 70)